In [26]:
import pandas as pd
import numpy as np

In [27]:
dataset_1 = pd.read_csv('../data/2024-09-08_2024-09-14_Win64_train_with_nulls.csv')

In [28]:
dataset_1.shape

(20000, 32)

In [29]:
dataset_1['family'].isnull().sum()

np.int64(12072)

In [30]:
dataset_1['family_confidence'].isnull().sum()

np.int64(12072)

family                   12072
family_confidence        12072

In [31]:
dataset_1.columns

Index(['md5', 'sha1', 'sha256', 'tlsh', 'first_submission_date',
       'last_analysis_date', 'detection_ratio', 'label', 'file_type', 'family',
       'family_confidence', 'behavior', 'file_property', 'packer', 'exploit',
       'group', 'histogram', 'byteentropy', 'strings', 'general', 'header',
       'section', 'imports', 'exports', 'datadirectories', 'richheader',
       'authenticode', 'pefilewarnings', 'week_id', 'caps', 'ttps', 'mbc'],
      dtype='str')

In [32]:
dataset_1['sha256'].describe(include=all)

/var/folders/25/kpc_zts11rs285txdrf5h0sh0000gn/T/ipykernel_11491/1643659544.py:1: Pandas4Warning: The 'include' and 'exclude' arguments are deprecated for Series.describe and will be removed in a future version. These arguments have no effect on Series and will be removed.
  dataset_1['sha256'].describe(include=all)


count                                                 20000
unique                                                10000
top       00047dd8d224e115c979a9eb1acd116e7a4a8a72f02594...
freq                                                      2
Name: sha256, dtype: object

In [33]:
dataset_1['sha256'].value_counts().value_counts()

count
2    10000
Name: count, dtype: int64

In [34]:
duplicate_rows = dataset_1[
    dataset_1['sha256'].duplicated(keep=False)
]

print(
    duplicate_rows.groupby('sha256')
    .nunique()
    .describe()
)

           md5     sha1     tlsh  first_submission_date  last_analysis_date  \
count  10000.0  10000.0  10000.0                10000.0             10000.0   
mean       1.0      1.0      1.0                    1.0                 1.0   
std        0.0      0.0      0.0                    0.0                 0.0   
min        1.0      1.0      1.0                    1.0                 1.0   
25%        1.0      1.0      1.0                    1.0                 1.0   
50%        1.0      1.0      1.0                    1.0                 1.0   
75%        1.0      1.0      1.0                    1.0                 1.0   
max        1.0      1.0      1.0                    1.0                 1.0   

       detection_ratio    label  file_type        family  family_confidence  \
count          10000.0  10000.0    10000.0  10000.000000       10000.000000   
mean               1.0      1.0        1.0      0.396400           0.396400   
std                0.0      0.0        0.0      0.4

In [35]:
print(dataset_1.duplicated().sum())

2958


In [36]:
dataset_1['sha256'].value_counts().idxmax()

'00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2'

In [37]:
duplicate_rows = dataset_1[
    dataset_1['sha256'].duplicated(keep=False)
]

variation = (
    duplicate_rows.groupby('sha256')
    .nunique()
)

print(
    variation.gt(1).sum()
    .sort_values(ascending=False)
)

caps                     7019
mbc                      6591
ttps                     6276
byteentropy                 0
week_id                     0
pefilewarnings              0
authenticode                0
richheader                  0
datadirectories             0
exports                     0
imports                     0
section                     0
header                      0
general                     0
strings                     0
md5                         0
sha1                        0
group                       0
exploit                     0
packer                      0
file_property               0
behavior                    0
family_confidence           0
family                      0
file_type                   0
label                       0
detection_ratio             0
last_analysis_date          0
first_submission_date       0
tlsh                        0
histogram                   0
dtype: int64


In [38]:
label_conflicts = (
    duplicate_rows.groupby('sha256')['label']
    .nunique()
)

print("Conflicting labels:",
      (label_conflicts > 1).sum())

Conflicting labels: 0


In [39]:
for col in ['md5', 'sha1']:
    result = duplicate_rows.groupby('sha256')[col].nunique()

    print(col)
    print("Conflicting groups:", (result > 1).sum())

md5
Conflicting groups: 0
sha1
Conflicting groups: 0


In [40]:
sample_hash = variation[
    variation.gt(1).any(axis=1)
].index[0]

print(
    duplicate_rows[
        duplicate_rows['sha256'] == sample_hash
    ].T
)

                                                                   0      \
md5                                     9c6701567cbcf50e11810c0a1ca2a580   
sha1                            43e53f8e44ecd49b55c8f5ed9ae33b8dd8e9534d   
sha256                 00047dd8d224e115c979a9eb1acd116e7a4a8a72f02594...   
tlsh                   T1B9C6338D77150C9DE95FA239E4E1863766B375B50BA0...   
first_submission_date                                         1726320821   
last_analysis_date                                            1730346557   
detection_ratio                                                    12/73   
label                                                                  1   
file_type                                                          Win64   
family                                                               NaN   
family_confidence                                                    NaN   
behavior                                                              []   
file_propert

In [41]:
# Check how many records have empty/non-empty
# CAPS, TTPS and MBC values

for col in ['caps', 'ttps', 'mbc']:

    print(f"\n{col.upper()}")

    print("Empty:",
          dataset_1[col].apply(
              lambda x: len(x) == 0 if isinstance(x, list) else pd.isna(x)
          ).sum())

    print("Non-empty:",
          dataset_1[col].apply(
              lambda x: len(x) > 0 if isinstance(x, list) else False
          ).sum())


CAPS
Empty: 0
Non-empty: 0

TTPS
Empty: 0
Non-empty: 0

MBC
Empty: 0
Non-empty: 0


In [42]:
for col in ['caps', 'ttps', 'mbc']:

    print("\nCOLUMN:", col)

    print("Data types:")
    print(dataset_1[col].apply(type).value_counts())

    print("\nSample values:")
    print(dataset_1[col].head(5).tolist())


COLUMN: caps
Data types:
caps
<class 'str'>    20000
Name: count, dtype: int64

Sample values:
['[]', "[{'Capability': 'Find graphical window', 'Namespace': 'host-interaction/gui/window/find', 'Addrs': []}, {'Capability': 'Create or open mutex on windows', 'Namespace': 'host-interaction/mutex', 'Addrs': []}, {'Capability': 'Link function at runtime on windows', 'Namespace': 'linking/runtime-linking', 'Addrs': []}, {'Capability': 'Encode data using xor', 'Namespace': 'data-manipulation/encoding/xor', 'Addrs': ['0x140017f30', '0x140017f60', '0x140017f90', '0x140017fc0', '0x140017ff0', '0x140018030', '0x14006b5a0']}, {'Capability': 'Get file attributes', 'Namespace': 'host-interaction/file-system/meta', 'Addrs': ['0x14007c500']}, {'Capability': 'Create thread', 'Namespace': 'host-interaction/thread/create', 'Addrs': ['0x14007d3c0']}, {'Capability': 'Accept command line arguments', 'Namespace': 'host-interaction/cli', 'Addrs': ['0x140002cd0']}, {'Capability': 'Check mutex on windows', 'Na

In [43]:
import ast
import pandas as pd

for col in ['caps', 'ttps', 'mbc']:

    dataset_1[col] = dataset_1[col].apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str) else x
    )

    empty = dataset_1[col].apply(
        lambda x: len(x) == 0
    ).sum()

    non_empty = dataset_1[col].apply(
        lambda x: len(x) > 0
    ).sum()

    print(f"\n{col.upper()}")
    print("Empty:", empty)
    print("Non-empty:", non_empty)


CAPS
Empty: 12981
Non-empty: 7019

TTPS
Empty: 13724
Non-empty: 6276

MBC
Empty: 13409
Non-empty: 6591


In [44]:
for col in ['caps', 'ttps', 'mbc']:

    print(f"\n{col.upper()}")

    result = dataset_1.groupby('sha256')[col].apply(
        lambda x: [
            len(v) for v in x
        ]
    )

    print(result.head(10))


CAPS
sha256
00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2    [0, 30]
0006d1a82c9c2ad4dfbbe34772b104bc889b4c0ebdcc9eaf94f82d2eb1dd0291    [25, 0]
001aea7ddef769f8b44ffc497cdf4f05193b4889510a5385a5d778817a09f8c3     [4, 0]
001b30621be4731fbd66ce0480ffcf15f8feeebedd9b159352c24ff146b718cc     [1, 0]
001dc027ca8aea62fdf19327d2a74ddf156aef93c8cbb0dab6b4ba728d475bc9     [2, 0]
001e3da23ae36c7ef6af5850f469234b06fc40779ec8f43749465ec0351ed467     [5, 0]
00208499e4f2a02320ebdd993446ecd310b026e19cfebb38eec60b73dcb5815f    [0, 29]
00215869134d30dbe58d0c9c3b462ac33590b0552bf71af964a9a76842e0163a    [0, 27]
0022831516cb4d9e122c17cba922efdccfef0ceaaea9d4eef34a629574d35cc3    [31, 0]
0022df1d0967d5de37784ae774e7447b6327fa509aeb87a9084aa4f371921c0f     [2, 0]
Name: caps, dtype: object

TTPS
sha256
00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2    [0, 14]
0006d1a82c9c2ad4dfbbe34772b104bc889b4c0ebdcc9eaf94f82d2eb1dd0291    [11, 0]
001aea7ddef769f8b44ffc497cdf4f05193b

In [45]:
for col in ['caps', 'ttps', 'mbc']:

    stats = dataset_1.groupby('sha256')[col].apply(
        lambda x: sum(len(v) > 0 for v in x)
    )

    print(f"\n{col.upper()}")
    print("Both empty:", (stats == 0).sum())
    print("One enriched:", (stats == 1).sum())
    print("Both enriched:", (stats == 2).sum())


CAPS
Both empty: 2981
One enriched: 7019
Both enriched: 0

TTPS
Both empty: 3724
One enriched: 6276
Both enriched: 0

MBC
Both empty: 3409
One enriched: 6591
Both enriched: 0


In [46]:
# Check all three enrichment fields together

summary = dataset_1.groupby('sha256').agg(
    caps_count=('caps', lambda x: sum(len(v) > 0 for v in x)),
    ttps_count=('ttps', lambda x: sum(len(v) > 0 for v in x)),
    mbc_count=('mbc', lambda x: sum(len(v) > 0 for v in x))
)

all_three = (
    (summary['caps_count'] == 1) &
    (summary['ttps_count'] == 1) &
    (summary['mbc_count'] == 1)
)

print("All three enriched:", all_three.sum())

print("At least one enriched:",
      ((summary[['caps_count', 'ttps_count', 'mbc_count']] > 0)
       .any(axis=1)).sum())

print("All three empty:",
      ((summary[['caps_count', 'ttps_count', 'mbc_count']] == 0)
       .all(axis=1)).sum())

All three enriched: 6085
At least one enriched: 7042
All three empty: 2958


In [47]:
import pandas as pd
import hashlib
import json

def normalize_value(x):

    if isinstance(x, (list, dict)):
        return json.dumps(
            x,
            sort_keys=True,
            default=str
        )

    if pd.isna(x):
        return "<NULL>"

    return str(x)


conflict_results = {}

for col in dataset_1.columns:

    if col == 'sha256':
        continue

    temp = dataset_1[['sha256', col]].copy()

    temp['_normalized'] = temp[col].apply(
        normalize_value
    )

    conflicts = (
        temp.groupby('sha256')['_normalized']
        .nunique()
    )

    count = (conflicts > 1).sum()

    conflict_results[col] = count

    if count > 0:
        print(f"{col}: {count} conflicting groups")

print("\nTotal columns checked:", len(conflict_results))

caps: 7019 conflicting groups
ttps: 6276 conflicting groups
mbc: 6591 conflicting groups

Total columns checked: 31


In [48]:
print("Shape:", dataset_1.shape)

print("\nColumns:")
for i, col in enumerate(dataset_1.columns, 1):
    print(i, col)

print("\nFile types:")
print(dataset_1['file_type'].value_counts(dropna=False))

print("\nLabels:")
print(dataset_1['label'].value_counts(dropna=False))

Shape: (20000, 32)

Columns:
1 md5
2 sha1
3 sha256
4 tlsh
5 first_submission_date
6 last_analysis_date
7 detection_ratio
8 label
9 file_type
10 family
11 family_confidence
12 behavior
13 file_property
14 packer
15 exploit
16 group
17 histogram
18 byteentropy
19 strings
20 general
21 header
22 section
23 imports
24 exports
25 datadirectories
26 richheader
27 authenticode
28 pefilewarnings
29 week_id
30 caps
31 ttps
32 mbc

File types:
file_type
Win64    20000
Name: count, dtype: int64

Labels:
label
1    10000
0    10000
Name: count, dtype: int64


Phase 2 – Detailed Duplicate Conflict Investigation

Step 1. Generate a complete conflict summary
We already calculated conflict_results. Instead of repeating the entire analysis, let's convert its results into a readable DataFrame.
What are we doing?
Operation	Simple explanation
nunique()	Counts unique malware file hashes
DataFrame()	Converts dictionary results into a table
conflict_percentage	Calculates percentage of groups with differences
sort_values()	Places columns with the most conflicts first
display()	Displays the complete report

Example: If a column has 7,019 conflicting groups out of 10,000, its conflict percentage is 70.19%.

In [49]:
# Total unique SHA256 groups
total_groups = dataset_1['sha256'].nunique()

# Convert conflict results into DataFrame
conflict_summary = pd.DataFrame(
    list(conflict_results.items()),
    columns=['column', 'conflicting_groups']
)

# Calculate percentage of conflicting groups
conflict_summary['conflict_percentage'] = (
    conflict_summary['conflicting_groups']
    / total_groups * 100

).round(2)

# Sort by highest conflicts
conflict_summary = conflict_summary.sort_values(
    by='conflicting_groups',
    ascending=False
)

print("Total SHA256 groups:", total_groups)

display(conflict_summary)


Total SHA256 groups: 10000


,column,conflicting_groups,conflict_percentage
28,caps,7019,70.19
30,mbc,6591,65.91
29,ttps,6276,62.76
16,byteentropy,0,0.00
27,week_id,0,0.00
26,pefilewarnings,0,0.00
25,authenticode,0,0.00
24,richheader,0,0.00
23,datadirectories,0,0.00
22,exports,0,0.00


Step 2. Separate conflicting and non-conflicting columns
This will help us identify:
- Which columns need further investigation.
- Which columns are consistent between duplicate records.
- Which metadata can potentially be retained without conflict.

In [50]:
conflicting_columns = conflict_summary[
    conflict_summary['conflicting_groups'] > 0
]

consistent_columns = conflict_summary[
    conflict_summary['conflicting_groups'] == 0
]

print("Conflicting columns:")
display(conflicting_columns)

print("\nConsistent columns:")
display(consistent_columns)

Conflicting columns:


,column,conflicting_groups,conflict_percentage
28,caps,7019,70.19
30,mbc,6591,65.91
29,ttps,6276,62.76



Consistent columns:


,column,conflicting_groups,conflict_percentage
16,byteentropy,0,0.0
27,week_id,0,0.0
26,pefilewarnings,0,0.0
25,authenticode,0,0.0
24,richheader,0,0.0
23,datadirectories,0,0.0
22,exports,0,0.0
21,imports,0,0.0
20,section,0,0.0
19,header,0,0.0


Conflict audit – confirmed results
Audit completed
- Total records: 20,000
- Unique SHA-256 groups: 10,000
- Records per SHA-256: 2
- Columns checked: 31 (excluding SHA-256 itself)
Conflicting columns
Column	Conflicting groups	Percentage
CAPS	  7,019	             70.19%
MBC	      6,591	             65.91%
TTPS	  6,276	             62.76%

Percentage of SHA-256 groups with different values in the three enrichment fields.

Consistent columns
All remaining 28 columns have zero conflicting groups.

These include:
- md5, sha1, tlsh
- first_submission_date
- last_analysis_date
- detection_ratio
- label
- file_type
- family, family_confidence
- behavior, file_property
- packer, exploit, group
- histogram, byteentropy, strings
- general, header, section
- imports, exports, datadirectories
- richheader, authenticode, pefilewarnings
- week_id
Important observation: The conflicts are concentrated in CAPS, TTPS and MBC. This suggests that these three enrichment fields require special treatment during duplicate consolidation.

It does not necessarily mean that the records are incorrect.

2. Next step: Investigate actual conflicting values
Before merging duplicate records, we need to understand exactly what differs.

We already know how many groups conflict. Now we need to identify:
1. Are both records empty?
2. Is one record empty and the other populated?
3. Do both records contain different populated values?
4. Are the values complementary or contradictory?
This is important because we don't want to lose malware intelligence during preprocessing.

Step 1 – Inspect three conflicting groups from each column
Why are we doing this?
We are examining actual values instead of relying only on numerical summaries.

For example:
- If Record 1 contains [] and Record 2 contains 20 entries, we may be able to combine the information.
- If both contain different entries, we need to determine whether both sets should be retained.
- If the entries contradict each other, we need a different strategy.
We will also compare labels and detection ratios to make sure duplicate records don't have contradictory classification information.

In [51]:
# Investigate actual conflicting values

def inspect_conflicts(column, sample_size=3):

    print(f"\n{'='*70}")
    print(f"INVESTIGATING COLUMN: {column.upper()}")
    print(f"{'='*70}")

    temp = dataset_1[['sha256', column]].copy()

    temp['_normalized'] = temp[column].apply(
        normalize_value
    )

    conflicts = (
        temp.groupby('sha256')['_normalized']
        .nunique()
    )

    conflict_hashes = conflicts[
        conflicts > 1
    ].index[:sample_size]

    for sha in conflict_hashes:

        records = dataset_1[
            dataset_1['sha256'] == sha
        ]

        print(f"\nSHA256: {sha}")
        print("Number of records:", len(records))

        print("\nLabels:")
        print(records['label'].tolist())

        print("\nDetection ratios:")
        print(records['detection_ratio'].tolist())

        print("\nFirst submission dates:")
        print(records['first_submission_date'].tolist())

        print("\nLast analysis dates:")
        print(records['last_analysis_date'].tolist())

        print(f"\n{column.upper()} values:")

        for index, value in enumerate(
            records[column].tolist(), 1
        ):

            print(f"\nRecord {index}:")

            print(
                json.dumps(
                    value,
                    indent=2,
                    default=str
                )[:1500]
            )


for col in ['caps', 'ttps', 'mbc']:

    inspect_conflicts(col, sample_size=3)


INVESTIGATING COLUMN: CAPS

SHA256: 00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2
Number of records: 2

Labels:
[1, 1]

Detection ratios:
['12/73', '12/73']

First submission dates:
[1726320821, 1726320821]

Last analysis dates:
[1730346557, 1730346557]

CAPS values:

Record 1:
[]

Record 2:
[
  {
    "Capability": "Link function at runtime on windows",
    "Namespace": "linking/runtime-linking",
    "Addrs": []
  },
  {
    "Capability": "Create process suspended",
    "Namespace": "host-interaction/process/create",
    "Addrs": [
      "0x1400089e0"
    ]
  },
  {
    "Capability": "Create process on windows",
    "Namespace": "host-interaction/process/create",
    "Addrs": [
      "0x1400089e0"
    ]
  },
  {
    "Capability": "Encode data using xor",
    "Namespace": "data-manipulation/encoding/xor",
    "Addrs": [
      "0x140009810",
      "0x140009810",
      "0x140009810",
      "0x140009810",
      "0x140009810",
      "0x140009810"
    ]
  },
  {
    "Capa

conclusion - can read the CAPS investigation output. There is an important observation here.

For the SHA-256 group shown:
Attribute	Record 1	Record 2
Label	1	1
Detection ratio	12/73	12/73
First submission	Same	Same
Last analysis	Same	Same
CAPS	[]	[]

Both CAPS values are empty.

However, our earlier conflict audit reported 7,019 conflicting CAPS groups.

That creates an inconsistency: if both values are genuinely empty, this particular SHA-256 group should not be classified as conflicting in CAPS.

We should investigate this before making any consolidation decisions. We don't want to merge data based on an incorrect conflict calculation.

Step 1 – Verify the conflict calculation again
What we are checking
- Whether the current DataFrame contains the same values as the earlier audit.
- Whether the normalized values actually differ.
- Whether notebook execution order caused outdated results.
- Whether the first reported conflicting group is genuinely conflicting.

In [52]:
# Recheck CAPS conflicts using the current DataFrame

for col in ['caps', 'ttps', 'mbc']:

    print(f"\n{'='*60}")
    print(f"VERIFYING: {col.upper()}")
    print(f"{'='*60}")

    temp = dataset_1[['sha256', col]].copy()

    temp['_normalized'] = temp[col].apply(normalize_value)

    conflict_counts = (
        temp.groupby('sha256')['_normalized']
        .nunique()
    )

    conflict_hashes = conflict_counts[
        conflict_counts > 1
    ]

    print("Total SHA256 groups:", conflict_counts.size)
    print("Conflicting groups:", len(conflict_hashes))

    if len(conflict_hashes) > 0:

        sample_hash = conflict_hashes.index[0]

        sample = temp[
            temp['sha256'] == sample_hash
        ]

        print("\nFirst actual conflicting SHA256:")
        print(sample_hash)

        print("\nOriginal values:")

        for value in sample[col]:
            print("Value:", repr(value))
            print("Type:", type(value))

        print("\nNormalized values:")

        for value in sample['_normalized']:
            print(repr(value))

        print(
            "\nUnique normalized values:",
            sample['_normalized'].nunique()
        )


VERIFYING: CAPS
Total SHA256 groups: 10000
Conflicting groups: 7019

First actual conflicting SHA256:
00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2

Original values:
Value: []
Type: <class 'list'>
Value: [{'Capability': 'Link function at runtime on windows', 'Namespace': 'linking/runtime-linking', 'Addrs': []}, {'Capability': 'Create process suspended', 'Namespace': 'host-interaction/process/create', 'Addrs': ['0x1400089e0']}, {'Capability': 'Create process on windows', 'Namespace': 'host-interaction/process/create', 'Addrs': ['0x1400089e0']}, {'Capability': 'Encode data using xor', 'Namespace': 'data-manipulation/encoding/xor', 'Addrs': ['0x140009810', '0x140009810', '0x140009810', '0x140009810', '0x140009810', '0x140009810']}, {'Capability': 'Read file on windows', 'Namespace': 'host-interaction/file-system/read', 'Addrs': ['0x140001050', '0x140001210', '0x140001470', '0x140001600', '0x140001950', '0x140007cf0', '0x140007f90', '0x1400176ec', '0x14001b4b4', '0x1400

Conclusion - What the latest output tells us
CAPS conflict verified
For the first SHA-256 group:

CAPS
Record	Value
Record 1	[]
Record 2	Populated list containing capability information

TTPS
Record	Value
Record 1	[]
Record 2	Populated list containing ATT&CK behaviour information

This is an important distinction.

These are not necessarily contradictory records. They appear to contain complementary enrichment information.

For example:
- Record 1 has no CAPS enrichment.
- Record 2 has CAPS enrichment.
- Both records refer to the same SHA-256 file.
We should preserve the populated intelligence rather than discard it.
2. Our earlier enrichment statistics
Based on your previous notebook outputs:
Column	Both empty	One enriched	Both enriched
CAPS	2,981	7,019	0
TTPS	3,724	6,276	0
MBC	3,409	6,591	0

This is encouraging.

It suggests that the conflicts are caused by missing enrichment in one duplicate record, rather than two records containing different populated enrichment values.

3. Next: Verify all three fields before consolidation
This checks the enrichment status without modifying the original dataset.

One caution: This classifies enrichment presence, not whether two populated lists are identical. We will separately validate that if any groups contain two populated values.

In [53]:
# Final enrichment conflict classification

for col in ['caps', 'ttps', 'mbc']:

    print(f"\n{'='*60}")
    print(f"{col.upper()} CONSOLIDATION AUDIT")
    print(f"{'='*60}")

    grouped = dataset_1.groupby('sha256')[col].apply(list)

    both_empty = 0
    one_enriched = 0
    both_enriched = 0

    for values in grouped:

        populated = [
            value for value in values
            if isinstance(value, list) and len(value) > 0
        ]

        if len(populated) == 0:
            both_empty += 1

        elif len(populated) == 1:
            one_enriched += 1

        else:
            both_enriched += 1

    print("Both empty:", both_empty)
    print("One enriched:", one_enriched)
    print("Both enriched:", both_enriched)

    print(
        "Total groups:",
        both_empty + one_enriched + both_enriched
    )


CAPS CONSOLIDATION AUDIT
Both empty: 2981
One enriched: 7019
Both enriched: 0
Total groups: 10000

TTPS CONSOLIDATION AUDIT
Both empty: 3724
One enriched: 6276
Both enriched: 0
Total groups: 10000

MBC CONSOLIDATION AUDIT
Both empty: 3409
One enriched: 6591
Both enriched: 0
Total groups: 10000


Conclusion - Excellent! All three enrichment audits passed


This confirms that our initial hypothesis was correct.
1. Final enrichment audit results
Enrichment field	Both empty	One enriched	Both enriched	Total
CAPS	2,981	7,019	0	10,000
TTPS	3,724	6,276	0	10,000
MBC	3,409	6,591	0	10,000

What this means
For every SHA-256 group:
- Both records have empty enrichment lists, or
- Exactly one record has populated enrichment information.
- No group has two populated records for the same enrichment field.
Therefore, we can safely preserve the populated list and avoid losing enrichment information.

Our consolidation rule:
1. Keep the first record as the base record.
2. For CAPS, TTPS and MBC, retain the non-empty list if one exists.
3. If both lists are empty, retain [].
4. Keep the original metadata.
5. Never overwrite dataset_1.
One important distinction: this validates enrichment presence, not whether the records are semantically correct. We will preserve the source values rather than invent or infer intelligence.

2. Next practical: Consolidate duplicate records
Create a new code cell in data_preprocessing_dataset_1.ipynb.
Step 1 – Consolidation code

In [54]:
# ==========================================
# CONSOLIDATE DUPLICATE SHA256 RECORDS
# ==========================================

enrichment_cols = ['caps', 'ttps', 'mbc']

consolidated_records = []

for sha256, group in dataset_1.groupby('sha256', sort=False):

    # Verify all non-enrichment columns are consistent
    for col in dataset_1.columns:

        if col in enrichment_cols or col == 'sha256':
            continue

        unique_values = group[col].apply(
            normalize_value
        ).nunique()

        if unique_values > 1:
            raise ValueError(
                f"Unexpected conflict in {col}, SHA256: {sha256}"
            )

    # Start with the first record
    record = group.iloc[0].to_dict()

    # Consolidate enrichment fields
    for col in enrichment_cols:

        values = group[col].tolist()

        populated = [
            value for value in values
            if isinstance(value, list) and len(value) > 0
        ]

        if len(populated) > 1:
            raise ValueError(
                f"Multiple enriched records found in {col}: {sha256}"
            )

        if len(populated) == 1:
            record[col] = populated[0]

        else:
            record[col] = []

    consolidated_records.append(record)


# Create a separate consolidated DataFrame
dataset_1_consolidated = pd.DataFrame(
    consolidated_records,
    columns=dataset_1.columns
)

print("Original shape:", dataset_1.shape)

print(
    "Consolidated shape:",
    dataset_1_consolidated.shape
)

print(
    "Unique SHA256:",
    dataset_1_consolidated['sha256'].nunique()
)

print(
    "Duplicate SHA256 remaining:",
    dataset_1_consolidated['sha256'].duplicated().sum()
)

Original shape: (20000, 32)
Consolidated shape: (10000, 32)
Unique SHA256: 10000
Duplicate SHA256 remaining: 0


In [55]:
# ==========================================
# CONSOLIDATION VALIDATION
# ==========================================

print("Original records:", len(dataset_1))

print(
    "Consolidated records:",
    len(dataset_1_consolidated)
)

print(
    "\nUnique SHA256:",
    dataset_1_consolidated['sha256'].nunique()
)

print(
    "Duplicate SHA256:",
    dataset_1_consolidated['sha256'].duplicated().sum()
)

print(
    "\nLabel distribution:"
)

print(
    dataset_1_consolidated['label'].value_counts(
        dropna=False
    )
)

print("\nEnrichment availability after consolidation:")

for col in ['caps', 'ttps', 'mbc']:

    populated = dataset_1_consolidated[col].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    ).sum()

    empty = len(dataset_1_consolidated) - populated

    print(f"\n{col.upper()}")
    print("Populated:", populated)
    print("Empty:", empty)

Original records: 20000
Consolidated records: 10000

Unique SHA256: 10000
Duplicate SHA256: 0

Label distribution:
label
1    5000
0    5000
Name: count, dtype: int64

Enrichment availability after consolidation:

CAPS
Populated: 7019
Empty: 2981

TTPS
Populated: 6276
Empty: 3724

MBC
Populated: 6591
Empty: 3409


Conclusion - Both outputs are correct. We have successfully completed duplicate SHA-256 consolidation without losing the enrichment data.
1. Validation results
Validation	Actual result	Status
Original records	20,000	Passed
Consolidated records	10,000	Passed
Original columns	32	Preserved
Unique SHA-256	10,000	Passed
Remaining duplicate SHA-256	0	Passed

2. Label distribution
Label	Records	Percentage
1	5,000	50%
0	5,000	50%

Perfectly balanced dataset
This is useful for our future malware classification model because neither class dominates the dataset.

However, balanced labels alone do not guarantee a good model.
3. Enrichment availability
Feature	Populated	Empty	Availability
CAPS	7,019	2,981	70.19%
TTPS	6,276	3,724	62.76%
MBC	6,591	3,409	65.91%

All three enrichment fields have retained their populated values.
4. What we have achieved
1. Identified 10,000 unique SHA-256 groups from 20,000 records.
2. Investigated conflicting enrichment fields.
3. Verified the actual conflicting values.
4. Confirmed that no two records had populated enrichment lists simultaneously for CAPS, TTPS or MBC.
5. Consolidated duplicate records into 10,000 records.
6. Validated labels and enrichment availability.

5. Next phase: Data Quality Assessment
Now we move to the next phase of our dataset preprocessing.

We will use dataset_1_consolidated, not the original dataset_1.

Our next tasks:
1. Missing-value analysis across all 32 columns.
2. Empty-list and null-value analysis.
3. Data type inspection.
4. Nested JSON structure inspection.
5. Timestamp and date validation.
6. Label and feature consistency.
7. Identify columns requiring preprocessing.

Start with Step 1: Missing-value analysis.

In [56]:
# ==========================================
# MISSING VALUE ANALYSIS
# ==========================================

missing_summary = pd.DataFrame({
    'missing_count': dataset_1_consolidated.isna().sum(),
    'missing_percentage': (
        dataset_1_consolidated.isna().mean() * 100
    )
})

missing_summary = missing_summary.sort_values(
    by='missing_percentage',
    ascending=False
)

print("MISSING VALUE SUMMARY")
display(missing_summary)

print(
    "\nTotal missing cells:",
    dataset_1_consolidated.isna().sum().sum()
)

print(
    "Total dataset cells:",
    dataset_1_consolidated.size
)

MISSING VALUE SUMMARY


,missing_count,missing_percentage
family,6036,60.36
family_confidence,6036,60.36
md5,0,0.00
general,0,0.00
header,0,0.00
section,0,0.00
imports,0,0.00
exports,0,0.00
datadirectories,0,0.00
byteentropy,0,0.00



Total missing cells: 12072
Total dataset cells: 320000


Conclusion - Excellent. Your missing-value analysis has completed successfully. The results are actually quite clean, and we have identified one important issue that needs investigation before preprocessing.
1. Understanding the results
Total dataset cells
320,000

Missing cells
12,072

Missing percentage
3.77%

Non-missing cells
307,928



Overall missingness



3.77% missing

96.23% available


Columns containing missing values
Column	Missing records	Percentage
family	6,036	60.36%
family_confidence	6,036	60.36%
Remaining 30 columns	0	0%

Important observation: Both family and family_confidence have exactly the same number of missing values.

This may indicate that family classification and its confidence score are unavailable together for the same malware samples.

We should verify this rather than assume it.
2. What should we do?
Do not delete these 6,036 records.

Reasons:
- They contain other potentially valuable malware intelligence.
- Their SHA-256 identifiers are available.
- They may contain CAPS, TTPS, MBC and other useful features.
- Missing family classification does not mean the sample is useless.
We should preserve the missing information and later decide how to handle it during feature engineering.

3. Next practical: Investigate missing values

In [57]:
# ==========================================
# INVESTIGATE MISSING FAMILY INFORMATION
# ==========================================

family_missing = dataset_1_consolidated[
    dataset_1_consolidated['family'].isna()
]

confidence_missing = dataset_1_consolidated[
    dataset_1_consolidated['family_confidence'].isna()
]

print("Family missing:", len(family_missing))

print(
    "Family confidence missing:",
    len(confidence_missing)
)

print(
    "Both missing:",
    (
        dataset_1_consolidated['family'].isna() &
        dataset_1_consolidated['family_confidence'].isna()
    ).sum()
)

print(
    "Family missing but confidence available:",
    (
        dataset_1_consolidated['family'].isna() &
        dataset_1_consolidated['family_confidence'].notna()
    ).sum()
)

print(
    "Family available but confidence missing:",
    (
        dataset_1_consolidated['family'].notna() &
        dataset_1_consolidated['family_confidence'].isna()
    ).sum()
)

Family missing: 6036
Family confidence missing: 6036
Both missing: 6036
Family missing but confidence available: 0
Family available but confidence missing: 0


4. Next: Check empty lists separately
One important technical distinction: isna() does not consider an empty list ([]) to be missing.

Our dataset contains nested fields such as CAPS, TTPS, MBC, behavior, imports and exports. We must inspect their actual contents.

In [58]:
# ==========================================
# EMPTY LIST ANALYSIS
# ==========================================

list_columns = [
    col for col in dataset_1_consolidated.columns
    if dataset_1_consolidated[col].apply(
        lambda x: isinstance(x, list)
    ).any()
]

for col in list_columns:

    empty_count = dataset_1_consolidated[col].apply(
        lambda x: isinstance(x, list) and len(x) == 0
    ).sum()

    print(
        f"{col}: {empty_count} empty lists "
        f"({empty_count / len(dataset_1_consolidated) * 100:.2f}%)"
    )

caps: 2981 empty lists (29.81%)
ttps: 3724 empty lists (37.24%)
mbc: 3409 empty lists (34.09%)


Conclusion -Excellent. Both outputs are correct. We have confirmed two important characteristics of our malware intelligence dataset.
1. Family missingness investigation
Investigation completed
Check	Result
Family missing	6,036
Family confidence missing	6,036
Both missing	6,036
Family missing, confidence available	0
Family available, confidence missing	0

Conclusion: family and family_confidence are perfectly aligned. Whenever family classification is unavailable, its confidence score is also unavailable.

We should preserve these records. Do not replace missing family names with invented malware families.
2. Empty-list analysis
Analysis completed
Enrichment field	Empty lists	Available lists
CAPS	2,981 (29.81%)	7,019
TTPS	3,724 (37.24%)	6,276
MBC	3,409 (34.09%)	6,591

These are not ordinary null values.
- [ ] means the field contains an empty list.
- NaN means the DataFrame value is missing.
- A populated list contains enrichment information.
For our platform, this distinction matters because we want to preserve evidence availability and avoid interpreting missing enrichment as proof that a malware sample has no malicious behavior.

4. Next phase: Data type and nested structure inspection
Now we need to understand exactly what Python objects are stored in our 32 columns.

This is especially important because our dataset contains nested JSON-like malware intelligence, not just ordinary numerical and categorical columns.
What we are looking for
- Numerical columns: integers and floats.
- Categorical columns: strings.
- Nested structures: lists and dictionaries.
- Timestamp fields: Unix timestamps or date strings.
- Enrichment fields: CAPS, TTPS and MBC structures.

In [59]:
# ==========================================
# DATA TYPE AND STRUCTURE INSPECTION
# ==========================================

print("DATAFRAME DATA TYPES")
print("=" * 60)

print(dataset_1_consolidated.dtypes)

print("\nPYTHON OBJECT TYPE DISTRIBUTION")
print("=" * 60)

for col in dataset_1_consolidated.columns:

    print(f"\nCOLUMN: {col}")

    type_counts = (
        dataset_1_consolidated[col]
        .dropna()
        .apply(lambda x: type(x).__name__)
        .value_counts()
    )

    print(type_counts)

print("\nDATASET MEMORY USAGE")
print("=" * 60)

print(
    dataset_1_consolidated.memory_usage(deep=True).sum()
    / (1024 ** 2),
    "MB"
)

DATAFRAME DATA TYPES
md5                          str
sha1                         str
sha256                       str
tlsh                         str
first_submission_date      int64
last_analysis_date         int64
detection_ratio              str
label                      int64
file_type                    str
family                       str
family_confidence        float64
behavior                     str
file_property                str
packer                       str
exploit                      str
group                        str
histogram                    str
byteentropy                  str
strings                      str
general                      str
header                       str
section                      str
imports                      str
exports                      str
datadirectories              str
richheader                   str
authenticode                 str
pefilewarnings               str
week_id                    int64
caps                  

Conclusion--Good. We have completed the first inspection of the consolidated dataset's data types and memory usage.

There are two important findings in your screenshots.
1. Initial findings
Inspection executed successfully
Column	Observed type
md5	String
sha1	String
sha256	String
tlsh	String
first_submission_date	int64
last_analysis_date	int64
detection_ratio	String
label	int64
family	String
family_confidence	float64
file_type	String
behavior	String
file_property	String
packer	String
exploit	String
group	String
histogram	String
byteentropy	String
strings	String
general	String
header	String
section	String
imports	String

The remaining columns need the same inspection.

Important observation: Several fields that logically contain structured malware intelligence are stored as strings. We need to determine whether these are ordinary text values or JSON-serialized lists and dictionaries.

We should not convert them blindly.
2. Dataset memory usage
Current consolidated DataFrame memory
214.08 MB
Reported by pandas using deep memory accounting
This is manageable for your MacBook Air M1 with 8 GB RAM, provided we avoid unnecessary copies of the complete DataFrame.

3. Next practical: Inspect actual stored values
Before converting anything, run this additional cell.
Why this additional inspection matters
For example, these three values are different:
Stored value	Meaning
[]	Empty list
"[]"	String containing JSON representation of an empty list
NaN	Missing DataFrame value

We need to distinguish them before feature engineering.

In [60]:
# ==========================================
# ACTUAL PYTHON TYPE AND VALUE INSPECTION
# ==========================================

import json

print("ACTUAL PYTHON TYPES")
print("=" * 60)

for col in dataset_1_consolidated.columns:

    print(f"\nCOLUMN: {col}")
    print("Pandas dtype:", dataset_1_consolidated[col].dtype)

    print(
        dataset_1_consolidated[col]
        .dropna()
        .apply(lambda x: type(x).__name__)
        .value_counts()
        .to_dict()
    )


# ==========================================
# INSPECT NESTED JSON-LIKE COLUMNS
# ==========================================

nested_columns = [
    'caps', 'ttps', 'mbc',
    'behavior', 'file_property',
    'packer', 'exploit', 'group',
    'histogram', 'byteentropy',
    'strings', 'general', 'header',
    'section', 'imports', 'exports',
    'datadirectories'
]

for col in nested_columns:

    if col not in dataset_1_consolidated.columns:
        continue

    print(f"\n{'=' * 50}")
    print("COLUMN:", col)

    samples = dataset_1_consolidated[col].dropna().head(3)

    for i, value in enumerate(samples, 1):

        print(f"\nSample {i}:")
        print("Python type:", type(value).__name__)
        print("Value:", repr(value)[:500])

        if isinstance(value, str):

            try:
                parsed = json.loads(value)
                print("JSON parsing: Successful")
                print("Parsed type:", type(parsed).__name__)

            except (json.JSONDecodeError, TypeError):
                print("JSON parsing: Not valid JSON")

ACTUAL PYTHON TYPES

COLUMN: md5
Pandas dtype: str
{'str': 10000}

COLUMN: sha1
Pandas dtype: str
{'str': 10000}

COLUMN: sha256
Pandas dtype: str
{'str': 10000}

COLUMN: tlsh
Pandas dtype: str
{'str': 10000}

COLUMN: first_submission_date
Pandas dtype: int64
{'int': 10000}

COLUMN: last_analysis_date
Pandas dtype: int64
{'int': 10000}

COLUMN: detection_ratio
Pandas dtype: str
{'str': 10000}

COLUMN: label
Pandas dtype: int64
{'int': 10000}

COLUMN: file_type
Pandas dtype: str
{'str': 10000}

COLUMN: family
Pandas dtype: str
{'str': 3964}

COLUMN: family_confidence
Pandas dtype: float64
{'float': 3964}

COLUMN: behavior
Pandas dtype: str
{'str': 10000}

COLUMN: file_property
Pandas dtype: str
{'str': 10000}

COLUMN: packer
Pandas dtype: str
{'str': 10000}

COLUMN: exploit
Pandas dtype: str
{'str': 10000}

COLUMN: group
Pandas dtype: str
{'str': 10000}

COLUMN: histogram
Pandas dtype: str
{'str': 10000}

COLUMN: byteentropy
Pandas dtype: str
{'str': 10000}

COLUMN: strings
Pandas dtype

Conclusion-- Excellent. This output reveals an important issue in Dataset 1.
1. What we have confirmed
- md5, sha1, sha256, tlsh are strings across all 10,000 consolidated records.
- first_submission_date and last_analysis_date are integers.
- At least one structured field contains Python-style dictionary/list text rather than valid JSON.

Your screenshot shows a value resembling:
"[{'has_relocs': 1, 'has_dynamic_relocs': 0}, ...]"
This fails json.loads() because Python-style representations use single quotes, whereas standard JSON requires double quotes.

This is not necessarily corrupted data. It may simply be serialized Python data.

We should use ast.literal_eval() to investigate it safely rather than using eval().

2. Next step: Identify serialization formats
Do not change the original DataFrame. Run this cell to classify the stored values across all relevant columns.
What we need from the output
We want to establish:
Category	What it tells us
Valid JSON	Can be parsed using json.loads()
Python literal	Can potentially be parsed using ast.literal_eval()
Already structured	Already a Python list/dictionary
Plain text	Should remain text unless further analysis says otherwise
Missing	Actual missing values

In [61]:

# ==========================================
# SERIALIZATION FORMAT AUDIT
# ==========================================

import ast
import json
import pandas as pd

def classify_value(x):

    if isinstance(x, (list, dict)):
        return "Already structured (Python object)"

    if pd.isna(x):
        return "Missing"

    if not isinstance(x, str):
        return "Other type"

    value = x.strip()

    if value == "":
        return "Empty string"

    # Check standard JSON
    try:
        json.loads(value)
        return "Valid JSON"
    except (json.JSONDecodeError, TypeError):
        pass

    # Check Python literal representation
    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, (list, dict, tuple)):
            return "Python literal (list/dict/tuple)"

        return "Python literal (scalar)"

    except (ValueError, SyntaxError, TypeError):
        return "Plain text / Other"

# Audit all columns without modifying the DataFrame

audit_results = []

for col in dataset_1_consolidated.columns:

    classifications = (
        dataset_1_consolidated[col]
        .apply(classify_value)
    )

    counts = classifications.value_counts()

    for category, count in counts.items():

        audit_results.append({
            "column": col,
            "category": category,
            "count": count,
            "percentage": round(
                count / len(dataset_1_consolidated) * 100, 2
            )
        })

serialization_audit = pd.DataFrame(audit_results)

print("SERIALIZATION FORMAT AUDIT")
print("=" * 70)

print(
    serialization_audit.to_string(index=False)
)

# ==========================================
# SHOW EXAMPLES OF PYTHON-LITERAL VALUES
# ==========================================

print("\nPYTHON LITERAL EXAMPLES")
print("=" * 70)

for col in dataset_1_consolidated.columns:

    mask = (
        dataset_1_consolidated[col]
        .apply(classify_value)
        .eq("Python literal (list/dict/tuple)")
    )

    if mask.any():

        print(f"\nCOLUMN: {col}")

        for value in dataset_1_consolidated.loc[
            mask, col
        ].head(2):

            print(repr(value)[:500])


SERIALIZATION FORMAT AUDIT
               column                           category  count  percentage
                  md5                 Plain text / Other  10000      100.00
                 sha1                 Plain text / Other  10000      100.00
               sha256                 Plain text / Other  10000      100.00
                 tlsh                 Plain text / Other  10000      100.00
first_submission_date                         Other type  10000      100.00
   last_analysis_date                         Other type  10000      100.00
      detection_ratio                 Plain text / Other  10000      100.00
                label                         Other type  10000      100.00
            file_type                 Plain text / Other  10000      100.00
               family                            Missing   6036       60.36
               family                 Plain text / Other   3964       39.64
    family_confidence                            Missing   60

In [62]:
dataset_1_consolidated.shape

(10000, 32)

In [65]:
dataset_1_consolidated['label'].value_counts()

label
1    5000
0    5000
Name: count, dtype: int64

First, preserve both datasets
Run this code in your notebook:

In [66]:

import os
import pandas as pd

# Create backup directory
os.makedirs("../data/processed", exist_ok=True)

# Preserve original dataset
dataset_1_raw = dataset_1.copy(deep=True)

# Preserve consolidated dataset
dataset_1_consolidated_backup = (
    dataset_1_consolidated.copy(deep=True)
)

# Save original 20,000 records
dataset_1_raw.to_pickle(
    "../data/processed/dataset_1_raw_20000.pkl"
)

# Save consolidated 10,000 records
dataset_1_consolidated_backup.to_pickle(
    "../data/processed/dataset_1_consolidated_10000.pkl"
)

# Also export CSV copies
dataset_1_raw.to_csv(
    "../data/processed/dataset_1_raw_20000.csv",
    index=False
)

dataset_1_consolidated_backup.to_csv(
    "../data/processed/dataset_1_consolidated_10000.csv",
    index=False
)

print("DATASET BACKUP COMPLETED")

print("Original:", dataset_1_raw.shape)
print("Consolidated:", dataset_1_consolidated_backup.shape)


DATASET BACKUP COMPLETED
Original: (20000, 32)
Consolidated: (10000, 32)


One final validation before moving forward

In [67]:

# Verify dataset integrity

assert len(dataset_1_raw) == 20000

assert len(dataset_1_consolidated) == 10000

assert dataset_1_consolidated["sha256"].nunique() == 10000

assert dataset_1_consolidated["sha256"].duplicated().sum() == 0

assert (
    dataset_1_consolidated["label"].value_counts().to_dict()
    == {1: 5000, 0: 5000}
)

print("All validation checks passed.")


All validation checks passed.


Conclusion-- Perfect! Both cells executed successfully. We have completed the backup and integrity validation.
Confirmed results
Check	Status
Original dataset: 20,000	Passed
Consolidated dataset: 10,000	Passed
Unique SHA256: 10,000	Passed
Remaining duplicate SHA256: 0	Passed
Label 0: 5,000	Passed
Label 1: 5,000	Passed
Backup verification	Passed

Our current progress
1. Dataset inspection and schema analysis — completed
2. SHA256 duplicate and conflict analysis — completed
3. Enrichment consolidation — completed
4. Missing-value and empty-list analysis — completed
5. Serialization format audit — performed
6. Original and consolidated dataset backup — completed
7. Dataset integrity validation — completed
8. Structured-field parsing and validation — next